# AA-YOLO Demo: Infrared Small Target Detection
This notebook demonstrates inference with AA-YOLO on infrared imagery.

In [ ]:
import sys
sys.path.insert(0, '..')
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Load Pre-trained Model
Load the AA-YOLOv7-tiny model trained on IRSTD-1k or SIRST.

In [ ]:
from models.experimental import attempt_load

# Choose weights: 'best_model_AA_YOLOv7t/irstd1k_best.pt' or 'best_model_AA_YOLOv7t/sirst_best.pt'
WEIGHTS = '../best_model_AA_YOLOv7t/irstd1k_best.pt'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = 640

model = attempt_load(WEIGHTS, map_location=DEVICE)
model.eval()
print(f"Model loaded on {DEVICE}")
print(f"Model type: {type(model.model[-1]).__name__}")

## 2. Inference on an Image
Define helper functions for preprocessing and postprocessing.

In [ ]:
from utils.general import non_max_suppression, scale_coords
from utils.datasets import letterbox

def preprocess(img_path, img_size=640):
    """Load and preprocess an image for inference."""
    img0 = cv2.imread(str(img_path))
    if img0 is None:
        raise FileNotFoundError(f"Image not found: {img_path}")
    img = letterbox(img0, img_size, stride=32)[0]
    img = img[:, :, ::-1].transpose(2, 0, 1)  # BGR to RGB, HWC to CHW
    img = np.ascontiguousarray(img)
    img = torch.from_numpy(img).to(DEVICE).float() / 255.0
    if img.ndimension() == 3:
        img = img.unsqueeze(0)
    return img, img0

def detect(model, img, conf_thres=0.1, iou_thres=0.05):
    """Run detection on preprocessed image."""
    with torch.no_grad():
        pred = model(img)[0]
    return non_max_suppression(pred, conf_thres, iou_thres, classes=None)

def draw_detections(img0, det, img_shape):
    """Draw bounding boxes on the image."""
    vis = img0.copy()
    if len(det):
        det[:, :4] = scale_coords(img_shape[2:], det[:, :4], img0.shape).round()
        for *xyxy, conf, cls in det:
            x1, y1, x2, y2 = map(int, xyxy)
            # Green box with confidence
            cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
            label = f'{conf:.2f}'
            cv2.putText(vis, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    return vis

## 3. Run Detection
Replace `IMAGE_PATH` with your infrared image path.

In [ ]:
# Replace with your image path
IMAGE_PATH = '../data/datasets/IRSTD-1k/images/test/XDU0.png'  # Example

try:
    img, img0 = preprocess(IMAGE_PATH, IMG_SIZE)
    detections = detect(model, img, conf_thres=0.1, iou_thres=0.05)
    
    print(f"Found {len(detections[0])} detections")
    if len(detections[0]):
        print(f"Confidences: {detections[0][:, 4].cpu().numpy()}")
    
    vis = draw_detections(img0, detections[0], img.shape)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(cv2.cvtColor(img0, cv2.COLOR_BGR2RGB))
    axes[0].set_title('Original')
    axes[0].axis('off')
    axes[1].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f'Detections ({len(detections[0])} targets)')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()
except FileNotFoundError as e:
    print(f"\u26a0\ufe0f {e}")
    print("Please update IMAGE_PATH to point to an actual infrared image.")

## 4. Visualize Anomaly Scores
Extract and visualize the anomaly-aware objectness maps from the AA detection head.

In [ ]:
def visualize_anomaly_maps(model, img):
    """Extract and display anomaly score maps from IDetect_AA head."""
    model.model[-1].training = True  # Get raw outputs
    with torch.no_grad():
        outputs = model(img)
    model.model[-1].training = False
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    scale_names = ['P3 (small)', 'P4 (medium)', 'P5 (large)']
    
    for idx, (out, name) in enumerate(zip(outputs, scale_names)):
        # Objectness is at index 4 (after xywh)
        obj_map = out[0, :, :, :, 4].mean(0).cpu().numpy()
        im = axes[idx].imshow(obj_map, cmap='hot', interpolation='nearest')
        axes[idx].set_title(f'{name} — Anomaly Objectness')
        axes[idx].axis('off')
        plt.colorbar(im, ax=axes[idx], fraction=0.046)
    
    plt.suptitle('AA-YOLO Anomaly Score Maps', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

try:
    visualize_anomaly_maps(model, img)
except Exception as e:
    print(f"\u26a0\ufe0f Visualization failed: {e}")
    print("This requires a loaded image (run cell 3 first).")

## 5. Batch Inference
Run detection on all images in a directory.

In [ ]:
def batch_detect(model, image_dir, img_size=640, conf_thres=0.1, iou_thres=0.05, max_images=10):
    """Run detection on multiple images and return results."""
    image_dir = Path(image_dir)
    results = []
    
    extensions = ['*.png', '*.jpg', '*.jpeg', '*.bmp']
    image_files = []
    for ext in extensions:
        image_files.extend(sorted(image_dir.glob(ext)))
    
    for img_path in image_files[:max_images]:
        img, img0 = preprocess(img_path, img_size)
        det = detect(model, img, conf_thres, iou_thres)
        n_det = len(det[0])
        results.append({'path': img_path.name, 'detections': n_det})
        print(f"  {img_path.name}: {n_det} targets detected")
    
    return results

# Example: batch inference on test set
# results = batch_detect(model, '../data/datasets/IRSTD-1k/images/test/', max_images=10)
print("Uncomment the line above to run batch inference on your dataset.")